In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("robot_arm_inverse_kinematics.ipynb")

# Robot arm - reach for a goal

Part 1: Use **fmin** to find joint angles that get the end location to a target. 

Slides (starts at 38ish): https://docs.google.com/presentation/d/1r7Aja2nI4iaLmaFbhugOf9mnopdR3ZFnkyVOKvLqKZc/edit?usp=sharing

In [ ]:
# The usual imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fmin

In [ ]:
# matrix_rouintes.py functions
import matrix_routines as mt

# These are the class you wrote in the previous lab/first part of the homework
from arm_component import ArmComponent
from robot_arm import RobotArm2D

In [ ]:
# These commands will force JN to actually re-load the external file when you re-execute the import command
%load_ext autoreload
%autoreload 2

# FMin optimization

In this problem you're going to write a function **distance_from_angles_for_fmin** that you will pass to fmin in order to have **fmin** find the angles that bring the end location of the last link to the target points

GUIDES: Edit **do_fmin** to call **func_for_fmin** with the appropriate angles, etc


In [ ]:
def distance_from_angles_for_fmin(angles : np.array, robot_arm:RobotArm2D, target):
    """ Compute the distance from the grasp point to the target
    @param angles as a numpy array, one angle for each joint (including palm)
    @param robot_arm - an instance of RobotArm2D
    @param target - a tuple with the desired x,y location
    @return The distance between the target and the gripper grasp point"""

    # GUIDES: step 1, call the set_link_angles method on robot_arm with the angle list
    # step 2, get the gripper location
    # step 3: calculate the distance between the target and the gripper

    # GUIDE: Call set_link_angles on the instance of robot_arm with the angles you just created
    #    This sets the matrix_pose's to the correct values for these angles
    ...

    # GUIDE: Get the end location and calculate the distance 

    ...
    ... # Return distance


In [ ]:
# Make an instance of the arm
base_size_param = (0.5, 1.0)   # Height, width
link_sizes_param = [(0.5, 0.25), (0.3, 0.1), (0.2, 0.05)]

# The sizes for all of the gripper components
palm_size = 0.1
finger_length = 0.075
grasp_percentage = 0.75

robot_arm = RobotArm2D(base_arm_size=base_size_param, link_sizes=link_sizes_param, palm_size=palm_size, finger_length=finger_length, grasp_percentage=grasp_percentage)

# Set the starting angles of the arm
angles_start = [np.pi/6.0, -np.pi/4, 1.5 * np.pi/4]
palm_angle = np.pi/3.0
finger_angle = np.pi/4.0
robot_arm.set_link_angles(angles_start, palm_angle=palm_angle, finger_angle=finger_angle)

# Get the end location
end_loc = robot_arm.get_gripper().get_grasp_location()

# Check that your function returns zero
#   This converts the start angles to a numpy array
angles_with_palm = [a for a in angles_start]
angles_with_palm.append(palm_angle)
angles_start_np = np.array(angles_with_palm)
dist = distance_from_angles_for_fmin(robot_arm=robot_arm, 
                                     angles=angles_start_np,                                      
                                     target=(end_loc[0], end_loc[1]))

assert np.isclose(dist, 0.0)

# Try some angles that should be further away
angles_not_start_as_np_array = angles_start_np * 0.5
dist_far_away = distance_from_angles_for_fmin(robot_arm=robot_arm, 
                                              angles=angles_not_start_as_np_array,                                      
                                              target=(end_loc[0], end_loc[1]))

assert dist_far_away > 0.0

In [ ]:
grader.check("optimization_dist_func")

## Do the fmin call

Here you're going to set up the **fmin** call. As always, you can write this outside of the function call then put it in the function after it's working

- First, convert the starting angles to a numpy array (see previous question)
- Next, call fmin with the function distance_from_angles_for_fmin and a tuple that has an instance of robot arm and the target in it


In [ ]:
def do_fmin(robot_arm:RobotArm2D, target):
    """ Set the angles/matrices of arm_geometry so they reach the target point
    @param - robot_arm - an instance of a robot arm
    @param - the target as a tuple (x,y)
    @return the robot arm with the angles set to the best ones
    """
    # GUIDES: 1: Get the current list of angles from the robot arm as a numpy array
    # GUIDES: 2: call fmin with the distance_from_angles_for_fmin function
    # GUIDES: 3: set the angles of the arm to the angles returned from distance_from_angles_for_fmin

    ...
    return robot_arm

In [ ]:
# The target
target = np.array([0.55, 1.15])

arm_geometry_optimized = do_fmin(robot_arm, target)

In [ ]:
assert np.isclose(robot_arm.get_gripper().get_grasp_location()[0], target[0], atol=0.01)
assert np.isclose(robot_arm.get_gripper().get_grasp_location()[1], target[1], atol=0.01)

In [ ]:
# Plot arm with target
fig, axs = plt.subplots(1, 1, figsize=(6, 6))
robot_arm.plot(axs=axs)
axs.plot(target[0], target[1], '+r', markersize=20)

In [ ]:
grader.check("do_fmin")

# Generalization

If nothing has been "hardwired" in, this should just work - changing the geometry, the starting angles, the target point. However, if you've hardwired something in, it probably won't...

In [ ]:
# Create another arm geometry
base_size_longer_param = (0.5, 0.25) # squished (height, width)
link_sizes_longer_param = [(0.3, 0.15), (0.2, 0.09), (0.1, 0.05), (0.075, 0.03)]
palm_width_longer_param = 0.15
finger_length_longer_param = (0.085, 0.015)


# Ths creates a different robot arm - one that is longer and with an additinal link
robot_arm_longer = RobotArm2D(base_size_longer_param, 
                                link_sizes_longer_param, 
                                palm_size=palm_size, 
                                finger_length=finger_length, 
                                grasp_percentage=0.5)

# Set the angles of the arm
angles_start_longer = [-np.pi/4.0, -np.pi/4, 1.2 * np.pi/4, -1 * np.pi/8]
robot_arm_longer.set_link_angles(angles_start_longer, palm_angle=-0.2, finger_angle=0.3)

target_longer = np.array([0.3, 0.15])

In [ ]:
# Plot arm with target
fig, axs = plt.subplots(1, 1, figsize=(8, 8))
robot_arm_longer.plot(axs)
axs.plot(target_longer[0], target_longer[1], '+r', markersize=20, label="target")

In [ ]:
# Do the optimization
arm_longer_optimized = do_fmin(robot_arm_longer, target_longer)

In [ ]:
# Plot arm with target
fig, axs = plt.subplots(1, 1, figsize=(8, 8))
arm_longer_optimized.plot(axs)
axs.plot(target_longer[0], target_longer[1], '+r', markersize=20, label="target")

In [ ]:
assert np.isclose(arm_longer_optimized.get_gripper().get_grasp_location()[0], target_longer[0], atol=0.01)
assert np.isclose(arm_longer_optimized.get_gripper().get_grasp_location()[1], target_longer[1], atol=0.01)


In [ ]:
grader.check("generalization")

## Numerical jacobian

The Jacobian is used to calculate how the end-effector moves as a joint angle changes. If there are 3 joint angles and the arm is in 2d, then this is a 2x3 matrix - each column is the derivative wrt that joint angle, each row is how x (first row) changes or how y (second row) changes

In this question you will calculate the jacobian numerically by **f(x+h) - f(x) / h**. This does not involve changing any code in *RobotArm.py**; just calling **set_link_angles** multiple times.

GUIDES: Edit robot_arm_ik_jacobian.py

In [ ]:
# This is for you to practice doing a Jacobian without the RobotArm class. You do not need to do it (it's not graded)
def practice_jacobian(b_print=False):
    """ An example problem of an arm with radius 3 currently at angle theta = 0.2
    This is just to check that you can calculate the Jacobian for one link before doing the matrix version
     If you have trouble with this, get the TA to give you the solution"""
    radius = 3
    theta = 0.2

    pt_end = [radius * np.cos(theta), radius * np.sin(theta)]

    # GUIDE: Create a 3D vector to the end point (r cos(theta), r sin(theta), 0)
    #   Needs to be 3D for cross product to work
    ...

    # The z vector we spin around
    omega_hat = [0, 0, 1]

    # GUIDE: Take the cross product of omega_hat and r
    #  The result should always be 0 in 3rd component for a 2D vector
    #  Order matters for cross products...
    ...


    # GUIDE: Build the Jacobian, which in this case is a 2x1 matrix
    # This matrix takes changes in the angles to changes in x,y
    #   ... and is just the first two components of omega hat cross r
    # GUIDE: Set the column of the matrix to be omega hat cross r
    jacobian_matrix = np.zeros([2, 1])
    ...

    # Now we'll set up a linear equation solve that looks like
    #  A x = b  (np.linalg.lstsq)
    # Where A is the Jacobian we just built, b is the desired change in x,y,
    #   and x is the angle we're solving form

    # GUIDE: Create a 2x1 matrix that stores the desired change in x and change in y
    #  ... in this case, use -0.01 and -0.1
    # Desired x,y change
    b_matrix = np.zeros([2, 1])
    ...
    pt_new_end = [pt_end[0] - 0.01, pt_end[1] - 0.1]

    # GUIDE: Solve the matrix using np.linalg.lstsq. Should return a 1x1 matrix with an angle change
    #   Note: Use rcond=None
    d_ang = np.zeros([1, 1])
    ...

    # Check result of solve - should be the same as dx_dy
    res = jacobian_matrix @ d_ang[0]

    # The actual point you end up at if you change the angle by that much
    pt_moved = [radius * np.cos(theta + d_ang[0][0]), radius * np.sin(theta + d_ang[0][0])]

    if b_print:
        print(f"Delta angle {d_ang[0]}, should be -0.32")
        print(f"New point and moved point should be the close to the same")
        print(f"Old point: {pt_end}\nNew pt: {pt_new_end}\nMoved pt: {pt_moved}")

    return d_ang


In [ ]:
# (NOT GRADED) Do this first to make sure you know how to calculate a Jacobian for 1 link
#  There's additional prints that print out the angle, moved point, etc.
print("Checking practice jacobian")
ang = practice_jacobian()
exp_ang = -0.032
try:
    if not np.isclose(ang, exp_ang, atol=0.01):
        print(f"Practice jacobian: Expected {exp_ang}, got {ang}")
    else:
        print(f"Passed practice jacobian test")
except:
    print(f"Probably an indexing/data structure problem Expected {exp_ang}, got {ang}\n")

In [ ]:
# Function you need to edit for this problem
from robot_arm_ik_jacobian import calculate_jacobian_numerically

In [ ]:
np.printoptions(precision=4)

# ----------------------------- Setup ------------------------
# These are example inputs for the create_arm_geometry function. 
base_size_param = (0.5, 1.0)   # Height, width
link_sizes_param = [(0.5, 0.25), (0.3, 0.1), (0.2, 0.05)]

# The sizes for all of the gripper components
palm_size = 0.1
finger_length = 0.075

robot_arm = RobotArm2D(base_arm_size=base_size_param, link_sizes=link_sizes_param, 
                        palm_size=palm_size, finger_length=finger_length)

angles_check = [np.pi/2, -np.pi/4, -3.0 * np.pi/4]
palm_angle = np.pi/3.0
finger_angle = np.pi/4.0

# The location to reach for
target = (0.5, 1.5)

# ------------------ Syntax and return value checks ----------------------

# First check - calculating the jacobian (2 x 4 matrix) calculated numerically
jacob_numerical = calculate_jacobian_numerically(robot_arm, angles_check, palm_angle=palm_angle)
exp_jacob = np.array([[-0.26084, -0.26084, -0.04871, -0.04871394],
                        [-0.48400, 0.016, 0.228, 0.02812]])
if not np.all(np.isclose(jacob_numerical, exp_jacob, atol=0.01)):
    print(f"Expected jacob\n{exp_jacob}, got\n{jacob_numerical}\n{np.isclose(jacob_numerical, exp_jacob, atol=0.01)}")
else:
    print("Passed numerical jacob test")

In [ ]:
grader.check("numerical_jacobian")

## Use the Jacobian to move the gripper closer to the goal

GUIDES: the code is in robot_arm_ik_jacobian.py, solve_jacobian. 

Extra credit: Change calculate_jacobian_numerically to calculate_jacobian in solve_jacobian, which does the Jacobian "properly"

In [ ]:
from robot_arm_ik_jacobian import calculate_jacobian, jacobian_follow_path, distance_to_goal

In [ ]:
# Optional: Calculate the Jacobian using partial derivatives
jacob_solve = calculate_jacobian(robot_arm, angles_check, palm_angle=palm_angle)
if not np.all(np.isclose(jacob_solve, exp_jacob, atol=0.01)):
    print(f"Expected jacob\n{exp_jacob}, got\n{jacob_solve}\n{np.isclose(jacob_solve, exp_jacob, atol=0.01)}")
else:
    print("Passed matrix jacob test")


In [ ]:
# ----------------- full solve check ----------------------------
# Check the run until no improvement version
b_succ, angles_new, count = jacobian_follow_path(robot_arm, angles_check, palm_angle=palm_angle, target=target, b_one_step=False)
robot_arm.set_link_angles(link_angles=angles_new[0:-1], palm_angle=angles_new[-1])
dist = distance_to_goal(robot_arm.get_gripper().get_grasp_location(), target)
if not b_succ:
    print(f"Expected successful/improvement, got none")
elif not np.isclose(dist, 0.062, atol=0.01):
    print(f"Expected distance to be close to 0.063, got {dist}, count {count}")
else:
    print(f"Passed full solve check, dist {dist:0.2}")


In [ ]:
# ----------------- Plot ----------------------------
# Plot so you can see what it should look like
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

angles_check.append(palm_angle)
for indx, a in enumerate([angles_check, angles_new]):
    # Actually get the matrices
    robot_arm.set_link_angles(a[0:-1], palm_angle=a[-1])
    robot_arm.plot(axs=axs[indx])

    # Plot target too
    axs[indx].plot(target[0], target[1], '+r')

axs[0].set_title("Before Jacobian descent")
axs[1].set_title(f"After Jacobian descent count {count} dist {dist:0.2}")


In [ ]:
grader.check("jacobian_descent")

## Hours and collaborators
Required for every assignment - fill out before you hand-in.

Listing names and websites helps you to document who you worked with and what internet help you received in the case of any plagiarism issues. You should list names of anyone (in class or not) who has substantially helped you with an assignment - or anyone you have *helped*. You do not need to list TAs.

Listing hours helps us track if the assignments are too long.

In [ ]:
import os

# List of names (creates a set)
worked_with_names = {"not filled out"}
# List of URLS FA26 (creates a set)
websites = {"not filled out"}
# Approximate number of hours, including lab/in-class time
hours = -1.5

In [ ]:
grader.check("hours_collaborators")

### To submit


- Submit this .ipynb file, arm_component.py, robot_arm.py, gripper.py, and robot_arm_ik_jacobian.py
- We will supply matrix_routines.py for you (it won't break anything if you do include it).
- As always, do a restart-runall before submitting and make sure your plots are visible.

If the Gradescope autograder fails, please check here first for common reasons for it to fail
    https://docs.google.com/presentation/d/1tYa5oycUiG4YhXUq5vHvPOpWJ4k_xUPp2rUNIL7Q9RI/edit?usp=sharing

Lots of people forget one or more of the .py files. Please check your autograder score to see if you are one of them.

Make sure you remove all the print statements you put in that print out lots of stuff.